In [ ]:
import hashlib, subprocess, pathlib
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab.')
subprocess.run(['git','clone','-q','https://github.com/kmeng01/rome.git','/content/rome'],check=True) if not pathlib.Path('/content/rome/.git').exists() else None
subprocess.run(['git','-C','/content/rome','checkout','-q','0874014cd9837e4365f3e6f3c71400ef11509e04'],check=True)
%pip -q install transformers==4.48.3 datasets==3.6.0 "accelerate>=0.26.0,<2" numpy scipy scikit-learn nltk matplotlib einops higher hydra-core
%cd /content/rome
import urllib.request
from pathlib import Path
Path('data').mkdir(exist_ok=True)
datasets = {'counterfact.json': 'd017056125178a13728594e66a801357a8db9ed7973a7425554bb4271de9fc6f',
            'known_1000.json': '61daca55318bb5260c1e62e133debef9102c7b278bbc7b19dd2ac655543f333a'}
for name, expected_hash in datasets.items():
    path = Path('data') / name
    if not path.exists():
        urllib.request.urlretrieve('https://rome.baulab.info/data/dsets/' + name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected_hash:
        raise ValueError(f'{path} differs from the published experiment input.')
from transformers import AutoModelForCausalLM, AutoTokenizer
revision = '15ea56dee5df4983c59b2538573817e1667135e2'
tok=AutoTokenizer.from_pretrained('openai-community/gpt2-xl',revision=revision)
tok.pad_token=tok.eos_token
model=AutoModelForCausalLM.from_pretrained('openai-community/gpt2-xl',revision=revision,torch_dtype=torch.float32,low_cpu_mem_usage=True).eval().cuda()
model.config._name_or_path='gpt2-xl'

In [ ]:
import json, contextlib, time, re
import numpy as np
from pathlib import Path
from rome import ROMEHyperParams, apply_rome_to_model
from baselines.ft import FTHyperParams, apply_ft_to_model
from experiments.py.eval_utils_counterfact import test_batch_prediction
from util import nethook
from transformers import set_seed
set_seed(42)
model.config.use_cache=False
out=Path('/content/rome_outputs');out.mkdir(exist_ok=True)
records=json.loads(Path('data/counterfact.json').read_text())[:7500]
perm=np.random.default_rng(42).permutation(len(records))
evaluation=[records[i] for i in perm[:200]]
pilot=[records[i] for i in perm[200:220]]
passages=['A folded sheet rested beside a shallow tray on a plain surface. The smallest crease ran from one corner toward the middle. Along the far edge a darker band shaded the paper. A small wooden bead rested in the tray. Its shadow reached just past a faint scratch. Beneath the sheet, a narrow strip of cloth lay flat. The weave was tighter near the edge and looser where the cloth had been folded. Light shifted across the surface while the objects stayed still. A faint ring marked the spot where a second tray had rested. The paper curled slightly at one corner, leaving a small gap above the cloth. From the side the gap looked like a thin line. The grain of the wood curved around a shallow dent.', 'A short row of tiles lined a narrow ledge. Each tile had a rough underside and a smooth face. A soft brush rested across the gap between two of them. Its bristles spread slightly where they touched the ledge. Near the handle a loose thread hung over the edge. A faint stripe crossed the middle tile and curved toward one corner. A smaller stripe stopped beside it. Seen from above, the pair looked almost parallel. The surface beneath the ledge had a shallow groove that filled with dust along one side. A small patch remained clear under the brush. Where the light struck the groove, its edge cast a narrow shadow. A speck near the end of the handle stayed in place as the shadow shortened.']
prefixes={}
for j,p in enumerate(passages):
    ids=tok.encode(p);assert len(ids)>=128
    for n in (32,128):
        prefix=tok.decode(ids[:n]);assert len(tok.encode(prefix))==n
        prefixes[f'{n}_{j+1}']=prefix
terms={r['requested_rewrite'][k]['str'] for r in evaluation+pilot for k in ['target_new','target_true']} | {r['requested_rewrite']['subject'] for r in evaluation+pilot}
collisions=[s for s in terms if any(re.search(r'(?<!\w)'+re.escape(s)+r'(?!\w)',p,re.I) for p in prefixes.values())]
assert not collisions, f'Overlapping prefix terms: {collisions}'
plan={'seed':42,'model_revision':revision,'rome_commit':'0874014cd9837e4365f3e6f3c71400ef11509e04','pilot':[x['case_id'] for x in pilot],'evaluation':[x['case_id'] for x in evaluation],'prefixes':prefixes}

def check_plan(name, expected, has_outputs):
    path = out / name
    if path.exists():
        if json.loads(path.read_text()) != expected:
            raise ValueError(f'{name} differs from the current run. Restore one complete checkpoint folder.')
    elif has_outputs:
        raise ValueError(f'Existing outputs are missing {name}. Restore the matching checkpoint folder.')
    else:
        with path.open('x') as stream:
            json.dump(expected, stream, indent=2)

def saved_case_files():
    return set(out.glob('eval_*.json')) | set(out.glob('pilot_*.json'))

case_files = saved_case_files()
expected_cases = {out / f'{phase}_{r["case_id"]:05d}.json'
                  for phase, rows in [('pilot', pilot), ('eval', evaluation)] for r in rows}
if case_files - expected_cases:
    raise ValueError('The checkpoint contains case files outside the current selection.')
has_outputs = case_files or any(out.glob('trace_*.npz')) or any((out / name).exists() for name in ('edit_contexts.json', 'trace_plan.json'))
check_plan('plan.json', plan, has_outputs)
methods={'ROME':(apply_rome_to_model,ROMEHyperParams.from_json('hparams/ROME/gpt2-xl.json')),'FT':(apply_ft_to_model,FTHyperParams.from_json('hparams/FT/gpt2-xl_unconstr.json')),'FT+L':(apply_ft_to_model,FTHyperParams.from_json('hparams/FT/gpt2-xl_constr.json'))}

In [ ]:
def score_record(r, extended):
    rw=r['requested_rewrite']; candidates=[rw['target_new']['str'],rw['target_true']['str']]
    groups={'rewrite':[rw['prompt'].format(rw['subject'])], 'paraphrase':r['paraphrase_prompts'], 'neighborhood':r['neighborhood_prompts']}
    output={}
    conditions={'0':''}; conditions.update(prefixes if extended else {})
    for label,pre in conditions.items():
        output[label]={}
        for group,items in groups.items():
            if label!='0' and group=='rewrite': continue
            prompts=[pre+'\n\n'+item if pre else item for item in items]
            assert max(len(tok.encode(p+' '+a)) for p in prompts for a in candidates)<=1024
            scores=[]
            for offset in range(0,len(prompts),3):
                scores+=test_batch_prediction(model,tok,prompts[offset:offset+3],*candidates)
            output[label][group]=scores
    return output

def do_case(record, phase):
    file=out/f"{phase}_{record['case_id']:05d}.json"
    if file.exists():
        saved = json.loads(file.read_text())
        if saved.get('case_id') != record['case_id'] or not {'unchanged', *methods, 'seconds', 'peak_gb'} <= saved.keys():
            raise ValueError(f'Incomplete or mismatched case in {file.name}.')
        return
    start=time.perf_counter(); torch.cuda.reset_peak_memory_stats()
    observed={'case_id':record['case_id'],'unchanged':score_record(record,True)}
    for name,(apply,params) in methods.items():
        with open('/content/rome-progress.log','a') as log, contextlib.redirect_stdout(log):
            _,orig=apply(model,tok,[record['requested_rewrite']],params,copy=False,return_orig_weights=True)
        try: observed[name]=score_record(record,name=='ROME')
        finally:
            with torch.no_grad():
                for param,backup in orig.items(): nethook.get_parameter(model,param).copy_(backup)
            del orig
    observed['seconds']=time.perf_counter()-start;observed['peak_gb']=torch.cuda.max_memory_allocated()/1e9
    temp=file.with_suffix('.tmp');temp.write_text(json.dumps(observed));temp.replace(file)
    print(phase,record['case_id'],round(observed['seconds'],1),'seconds',round(observed['peak_gb'],2),'GB',flush=True)
assert not collisions
import rome.rome_main as rome_main
context_path = out / 'edit_contexts.json'
length_params = methods['ROME'][1].context_template_length_params
if context_path.exists():
    contexts = json.loads(context_path.read_text())
    expected_count = 1 + sum(count for _, count in length_params)
    if (not isinstance(contexts, list) or len(contexts) != expected_count or contexts[0] != '{}'
            or not all(isinstance(s, str) and s.count('{}') == 1 for s in contexts)):
        raise ValueError('Invalid ROME context templates in edit_contexts.json.')
    rome_main.CONTEXT_TEMPLATES_CACHE = contexts
elif saved_case_files():
    raise ValueError('Existing edits require their saved edit_contexts.json.')
else:
    rome_main.CONTEXT_TEMPLATES_CACHE = None
    with open('/content/rome-progress.log','a') as log, contextlib.redirect_stdout(log):
        contexts = rome_main.get_context_templates(model, tok, length_params)
    with context_path.open('x') as stream:
        json.dump(contexts, stream)
for case in pilot: do_case(case,'pilot')
for case in evaluation: do_case(case,'eval')

In [ ]:
import math
from experiments.causal_trace import ModelAndTokenizer, calculate_hidden_flow, collect_embedding_std, make_inputs, predict_from_input, decode_tokens
model.requires_grad_(False);model.eval();mt=ModelAndTokenizer(model=model,tokenizer=tok)
known=json.loads(Path('data/known_1000.json').read_text());eligible=[]
with torch.no_grad():
    for fact in known:
        pred,_=predict_from_input(model,make_inputs(tok,[fact['prompt']]))
        if decode_tokens(tok,pred)[0].strip()==fact['attribute']: eligible.append(fact)
selected=[eligible[i] for i in np.random.default_rng(42).permutation(len(eligible))[:50]]
assert len(selected)==50
noise=3*collect_embedding_std(mt,[k['subject'] for k in known])
trace_plan = {'eligible':[f['known_id'] for f in eligible], 'selected':[f['known_id'] for f in selected], 'noise_std':noise}
trace_files = set(out.glob('trace_*.npz'))
expected_traces = {out / f'trace_{i}_{kind}.npz' for i in trace_plan['selected'] for kind in ('mlp', 'attn')}
if trace_files - expected_traces:
    raise ValueError('The checkpoint contains traces outside the current selection.')
trace_path = out / 'trace_plan.json'
if trace_path.exists():
    saved_plan = json.loads(trace_path.read_text())
    saved_noise = saved_plan.get('noise_std') if isinstance(saved_plan, dict) else None
    if (type(saved_noise) not in (float, int)
            or not math.isclose(saved_noise, noise, rel_tol=1e-5, abs_tol=1e-7)
            or saved_plan.keys() != trace_plan.keys()
            or saved_plan['eligible'] != trace_plan['eligible']
            or saved_plan['selected'] != trace_plan['selected']):
        raise ValueError('The saved trace plan differs from the current run.')
    noise = saved_noise
elif trace_files:
    raise ValueError('Existing traces require trace_plan.json.')
else:
    with trace_path.open('x') as stream:
        json.dump(trace_plan, stream, indent=2)
for fact in selected:
    for kind in ['mlp','attn']:
        dest=out/f"trace_{fact['known_id']}_{kind}.npz"
        if dest.exists():
            with np.load(dest, allow_pickle=False) as saved:
                if (saved['kind'].item() != kind or saved['window'].item() != 10 or not saved['correct_prediction'].item()
                        or ''.join(saved['input_tokens']) != fact['prompt'] or saved['answer'].item().strip() != fact['attribute']
                        or saved['scores'].shape != (len(saved['input_tokens']), mt.num_layers)):
                    raise ValueError(f'Incomplete or mismatched trace in {dest.name}.')
            continue
        result=calculate_hidden_flow(mt,fact['prompt'],fact['subject'],expect=fact['attribute'],samples=10,noise=noise,window=10,kind=kind)
        assert result['correct_prediction']
        arrays = {k: (v.detach().cpu().numpy() if torch.is_tensor(v) else v) for k, v in result.items()}
        temp = dest.with_suffix('.tmp')
        with temp.open('wb') as stream:
            np.savez_compressed(stream, **arrays)
        temp.replace(dest)
    print('traced',fact['known_id'],flush=True)